# Akan-BPE Model Ladder - Heavy Split

Runs the heavier/riskier half of the ladder: Qwen3-1.7B and Gemma-3-1B.

Run this top-to-bottom on Kaggle or Colab with a T4 GPU. Each model runs two arms: random embedding init and mean-of-subword embedding init. After each model, the notebook prints fertility/tokenization metrics, BPB metrics, generation samples, reload verification, the full JSON payloads, and then zips artifacts into `outputs/`.

## Setup

In [ ]:
# Clone the repo (skip if already inside it)
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

In [ ]:
!pip install -q uv
!uv pip install -q --system -e ".[dev,train]"
!uv pip install -q --system bitsandbytes sentencepiece

In [ ]:
# Hugging Face authentication. Paste a token with access to gated models
# when prompted. The input is hidden.
from huggingface_hub import login

login()

In [ ]:
# Confirm a GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable a T4 GPU accelerator, then re-run.")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Download Akan datasets. --tts-limit 50000 pins the 40,000/5,000/5,000 split.
!python scripts/download.py --output-dir data --tts-limit 50000

In [ ]:
# Verify required inputs exist. The TTS tokenizer is committed under models/.
from pathlib import Path

required = {
    "TTS train data": Path("data/pristine_twi_train.jsonl"),
    "TTS test data": Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer": Path("models/tts_tokenizer.json"),
}
missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")

## Reporting helpers

In [ ]:
import json
from pathlib import Path


def load_result(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing result JSON: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def fmt(value, digits=4):
    if value is None:
        return "n/a"
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


def result_paths(model_slug):
    return {
        "random": Path("results") / f"run-{model_slug}.json",
        "mean_subword": Path("results") / f"run-{model_slug}-meansub.json",
    }


def print_tokenization_table(results):
    print("\nTOKENIZATION / FERTILITY")
    print("tokens/word is fertility; lower is better.")
    header = (
        f"{'arm':<22}{'base fertility':>16}{'Akan fertility':>16}"
        f"{'base tokens':>14}{'Akan tokens':>14}{'words':>10}{'reduction':>12}"
    )
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        counts = payload["token_count_comparison"]
        base = counts["base_model_tokenizer"]
        akan = counts["experiment_tokenizer"]
        print(
            f"{arm:<22}"
            f"{fmt(base.get('fertility')):>16}"
            f"{fmt(akan.get('fertility')):>16}"
            f"{base.get('total_tokens'):>14}"
            f"{akan.get('total_tokens'):>14}"
            f"{akan.get('total_words'):>10}"
            f"{fmt(counts.get('token_reduction_ratio')):>12}"
        )


def print_bpb_table(results):
    print("\nMODELING / BPB")
    print("BPB uses full byte coverage; lower experiment BPB is better.")
    header = (
        f"{'arm':<22}{'eval_loss':>12}{'perplexity':>12}"
        f"{'experiment BPB':>18}{'base BPB':>12}{'improvement':>14}"
    )
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        bpb = payload["eval"]["bpb"]
        base_bpb = (bpb.get("base") or {}).get("bits_per_byte")
        print(
            f"{arm:<22}"
            f"{fmt(payload['eval'].get('eval_loss')):>12}"
            f"{fmt(payload['eval'].get('perplexity'), 2):>12}"
            f"{fmt(bpb['experiment'].get('bits_per_byte')):>18}"
            f"{fmt(base_bpb):>12}"
            f"{fmt(bpb.get('improvement')):>14}"
        )


def print_generation_samples(results):
    print("\nGENERATION SAMPLES")
    for arm, payload in results.items():
        print(f"\n[{arm}]")
        for index, sample in enumerate(payload.get("generation_samples", []), start=1):
            print(f"sample {index} prompt: {sample.get('prompt', '')}")
            print(f"sample {index} completion: {sample.get('completion', '')}")


def print_run_integrity(results):
    print("\nRUN INTEGRITY")
    header = f"{'arm':<22}{'output_model_dir':<34}{'reload ok':>10}{'device':>18}"
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        reload_info = payload.get("reload_verification") or {}
        device = payload.get("device") or {}
        print(
            f"{arm:<22}"
            f"{str(payload.get('output_model_dir')):<34}"
            f"{str(reload_info.get('success')):>10}"
            f"{str(device.get('device_name')):>18}"
        )


def print_full_json(results):
    for arm, payload in results.items():
        print(f"\nBEGIN_FULL_JSON {arm}")
        print(json.dumps(payload, ensure_ascii=False, indent=2))
        print(f"END_FULL_JSON {arm}")


def display_model_results(model_slug):
    paths = result_paths(model_slug)
    results = {arm: load_result(path) for arm, path in paths.items()}
    print("=" * 88)
    print(f"FULL RESULTS FOR {model_slug}")
    print("=" * 88)
    print("Result files:")
    for arm, path in paths.items():
        print(f"- {arm}: {path}")
    print_tokenization_table(results)
    print_bpb_table(results)
    print_generation_samples(results)
    print_run_integrity(results)
    best = min(results, key=lambda arm: results[arm]["eval"]["bpb"]["experiment"]["bits_per_byte"])
    print(f"\nBest downstream BPB arm: {best}")
    print_full_json(results)
    return results


## Run qwen-1.7b

Model id: `Qwen/Qwen3-1.7B-Base`. This section runs both arms, then immediately displays and archives the full results.

In [ ]:
# qwen-1.7b / Arm A - random embedding init
!python scripts/model_integration.py --model-id Qwen/Qwen3-1.7B-Base

In [ ]:
# qwen-1.7b / Arm B - mean-of-subword embedding init
!python scripts/model_integration.py --model-id Qwen/Qwen3-1.7B-Base --embedding-init-mode mean_subword

## Full results and artifact zip - qwen-1.7b

In [ ]:
results_qwen_1_7b = display_model_results("qwen-1.7b")

In [ ]:
!mkdir -p outputs
!ls -lh results models outputs
!zip -r outputs/run-qwen-1.7b-artifacts.zip     results/run-qwen-1.7b.json     results/run-qwen-1.7b-meansub.json     models/run-qwen-1.7b     models/run-qwen-1.7b-meansub
!ls -lh outputs

## Run gemma-1b

Model id: `google/gemma-3-1b-pt`. This section runs both arms, then immediately displays and archives the full results.

In [ ]:
# gemma-1b / Arm A - random embedding init
!python scripts/model_integration.py --model-id google/gemma-3-1b-pt

In [ ]:
# gemma-1b / Arm B - mean-of-subword embedding init
!python scripts/model_integration.py --model-id google/gemma-3-1b-pt --embedding-init-mode mean_subword

## Full results and artifact zip - gemma-1b

In [ ]:
results_gemma_1b = display_model_results("gemma-1b")

In [ ]:
!mkdir -p outputs
!ls -lh results models outputs
!zip -r outputs/run-gemma-1b-artifacts.zip     results/run-gemma-1b.json     results/run-gemma-1b-meansub.json     models/run-gemma-1b     models/run-gemma-1b-meansub
!ls -lh outputs

## Notebook-level summary

In [ ]:
MODEL_SLUGS = ['qwen-1.7b', 'gemma-1b']
all_results = {slug: {arm: load_result(path) for arm, path in result_paths(slug).items()} for slug in MODEL_SLUGS}

print("=" * 88)
print("NOTEBOOK SPLIT SUMMARY")
print("=" * 88)
print_tokenization_table({f"{slug}/{arm}": payload for slug, arms in all_results.items() for arm, payload in arms.items()})
print_bpb_table({f"{slug}/{arm}": payload for slug, arms in all_results.items() for arm, payload in arms.items()})

summary = {
    slug: {
        arm: {
            "eval_loss": payload["eval"]["eval_loss"],
            "perplexity": payload["eval"]["perplexity"],
            "experiment_bpb": payload["eval"]["bpb"]["experiment"]["bits_per_byte"],
            "base_bpb": (payload["eval"]["bpb"].get("base") or {}).get("bits_per_byte"),
            "bpb_improvement": payload["eval"]["bpb"].get("improvement"),
            "base_fertility": payload["token_count_comparison"]["base_model_tokenizer"]["fertility"],
            "akan_fertility": payload["token_count_comparison"]["experiment_tokenizer"]["fertility"],
            "token_reduction_ratio": payload["token_count_comparison"]["token_reduction_ratio"],
        }
        for arm, payload in arms.items()
    }
    for slug, arms in all_results.items()
}
Path("outputs").mkdir(exist_ok=True)
Path("outputs/heavy-split-summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote outputs/heavy-split-summary.json")


In [ ]:
!ls -lh outputs